# Bio 1B AI Usage & Grade Analysis

## Setup

In [1]:
# Loading Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [2]:
# Pull Data Sheets (No long drive paths needed!)
df_ap = pd.read_excel('EOSDataF25_DeID_AP&Grades.xlsx', sheet_name='F25_APBio')
df_grades = pd.read_excel('EOSDataF25_DeID_AP&Grades.xlsx', sheet_name='F25_Bio1BGrades')

df_grades = df_grades.drop(df_grades.index[-2:])
df_survey = pd.read_excel('EOSDataF25_DeID_AP&Grades.xlsx', sheet_name='DeID_Katy_NoDemo')

df_ap.head()

,UID,Term,Class Nbr,Subject,Catalog Nbr,Descr,Career,Major(s),Level (by Units),Terms in Attendance,...,Last Update Date,Last Update Time,Related 1 Class Nbr,Related 1 Section,Related 1 Component,Test ID,Descr.1,Component,Score,Test Dt
0,F25416,2258,21182,BIOLOGY,1B,GENERAL BIOLOGY,UGRD,Neuroscience BA,Senior,3,...,2025-04-24,0.500220,21171,221,LAB,NaN,NaN,NaN,0.0,NaT
1,F25335,2258,21182,BIOLOGY,1B,GENERAL BIOLOGY,UGRD,Letters & Sci Undeclared UG,Junior,3,...,2025-04-24,0.625150,21174,412,LAB,AP,English Literature & Compostn,AP37,3.0,2024-01-01
2,F25335,2258,21182,BIOLOGY,1B,GENERAL BIOLOGY,UGRD,Letters & Sci Undeclared UG,Junior,3,...,2025-04-24,0.625150,21174,412,LAB,AP,English Language & Composition,AP36,4.0,2023-01-01
3,F25335,2258,21182,BIOLOGY,1B,GENERAL BIOLOGY,UGRD,Letters & Sci Undeclared UG,Junior,3,...,2025-04-24,0.625150,21174,412,LAB,AP,English Language & Composition,AP36,4.0,2023-04-01
4,F25264,2258,21182,BIOLOGY,1B,GENERAL BIOLOGY,UGRD,Conserv & Resource Stds BS,Sophomore,3,...,2025-06-11,0.406123,21157,322,LAB,AP,English Literature & Compostn,AP37,5.0,2024-01-01


In [3]:
df_grades.shape # total number of students

(658, 49)

## Cleaning & Merging

#### Keeping only the AP/IB Biology scores


In [4]:
# Filter and deduplicate
bio_components = ["AP20", "IBH10", "IBS25"]

# Filter df_ap to ONLY include rows matching those components
df_ap_bio = df_ap[df_ap["Component"].isin(bio_components)].copy()

# Sort by Score descending so each student's highest score is at the top
# Deduplicate by student UID so each student appears exactly once
df_ap_bio = df_ap_bio.sort_values("Score", ascending=False).drop_duplicates("UID")

# Fix UID types
df_ap_bio["UID"] = df_ap_bio["UID"].astype(str)
df_grades["UID"] = df_grades["UID"].astype(str)

# Left join — keeps ALL students, NaN for those with no AP/IB Bio, keeping only Component and Score from grades table
joined_all = df_grades.merge(df_ap_bio[["UID", "Component", "Score"]], on="UID", how="left")

# Flag: did they take AP/IB Bio? (T/F)
joined_all["has_bio_background"] = joined_all["Score"].notna()

joined_all.head()

,UID,Section,Midterm 1 - Multiple Choice/Fixed Response Score (8916061),Midterm 1 - MC/fixed Response Make up (8916060),Midterm 1 - Free Response Score (8916059),Midterm 2 -Multiple Choice/Fixed Response Score (8916064),Midterm 2 - MC/Fixed Response Make up (8916063),Midterm 2 - Free Response (8916062),Midterm 3 - Multiple Choice/Fixed Response (8916066),Midterm 3 - Multiple Choice/Fixed Response Make Up (8916067),...,Final Score,Unposted Final Score,Current Grade,Unposted Current Grade,Final Grade,Unposted Final Grade,Unnamed: 48,Component,Score,has_bio_background
0,F25599,BIOLOGY 1B LAB 112 (In Person) and BIOLOGY 1B ...,43.68,53.63,4.5,54,48.75,6.5,54.0,48.75,...,75.09,75.09,C,C,C,C,NaN,NaN,NaN,False
1,F25490,BIOLOGY 1B LAB 221 (In Person) and BIOLOGY 1B ...,65.52,43.88,6.5,63,48.75,5.5,45.0,34.13,...,80.85,80.85,B-,B-,B-,B-,NaN,AP20,4.0,True
2,F25371,BIOLOGY 1B LAB 312 (In Person) and BIOLOGY 1B ...,78.00,0.00,8.0,75,0.00,6.5,63.0,58.50,...,93.26,93.26,A,A,A,A,NaN,AP20,4.0,True
3,F25195,BIOLOGY 1B LAB 221 (In Person) and BIOLOGY 1B ...,71.76,0.00,7.5,66,0.00,8.0,60.0,73.13,...,93.44,93.44,A,A,A,A,Cutoffs:,NaN,NaN,False
4,F25153,BIOLOGY 1B LAB 421 (In Person) and BIOLOGY 1B ...,0.00,NaN,0.0,0,NaN,0.0,0.0,NaN,...,1.74,1.74,F,F,F,F,NaN,NaN,NaN,False


## Exploratory Data Analysis

In [5]:
# Inital Look -- grouping by Exam Type, Score, Student Count, and Ratio over Total students count for each exam type

df_counts = joined_all.groupby(['Component', 'Score'])["UID"].count().reset_index().rename(columns={'UID': 'Student_Count'})
df_counts['Ratio'] = df_counts['Student_Count'] / df_counts.groupby('Component')['Student_Count'].transform('sum')
df_counts

,Component,Score,Student_Count,Ratio
0,AP20,1.0,10,0.031949
1,AP20,2.0,37,0.118211
2,AP20,3.0,82,0.261981
3,AP20,4.0,100,0.319489
4,AP20,5.0,84,0.268371
5,IBH10,3.0,6,0.375000
6,IBH10,4.0,6,0.375000
7,IBH10,5.0,4,0.250000
8,IBS25,3.0,2,0.222222
9,IBS25,4.0,3,0.333333


In [6]:
ap_bio_counts = df_counts.loc[df_counts["Component"] == "AP20", "Student_Count"].sum().item()
ap_bio_counts # Number of students that have taken AP Biology

313

In [7]:
ib_h_counts = df_counts.loc[df_counts["Component"] == "IBH10", "Student_Count"].sum().item()
ib_h_counts # Number of students that have taken IB HL Biology

16

In [8]:
ib_s_counts = df_counts.loc[df_counts["Component"] == "IBS25", "Student_Count"].sum().item()
ib_s_counts # Number of students that have taken IB SL Biology

9

In [9]:
312/658 # Around 47% of students have taken AP Bio

0.47416413373860183

In [10]:
# How do exam score averages compare between different groups?
joined_all = joined_all.rename(columns={'Final Exam Final Score (out of 126)': 'Final Exam Final Score'})

score = joined_all.groupby(['Component', 'Score'])[['Final Exam Final Score', 'Final Score']].mean().reset_index()
score = score.merge(
    df_counts[['Component', 'Score', 'Ratio']],
    on=['Component', 'Score'],
    how='left'
)
score["Percentage of Students"] = score["Ratio"] * 100
score = score.drop(columns=["Ratio"])
score = score.rename(columns = {"Final Exam Final Score":"Average Final Exam Final Score", "Final Score":"Average Final Score"})
score

,Component,Score,Average Final Exam Final Score,Average Final Score,Percentage of Students
0,AP20,1.0,59.762,78.704,3.194888
1,AP20,2.0,76.641622,84.213514,11.821086
2,AP20,3.0,82.985122,89.648902,26.198083
3,AP20,4.0,84.4296,91.0295,31.948882
4,AP20,5.0,91.072738,94.549524,26.837061
5,IBH10,3.0,75.791667,87.211667,37.500000
6,IBH10,4.0,83.73,89.195,37.500000
7,IBH10,5.0,82.74,91.345,25.000000
8,IBS25,3.0,63.095,81.465,22.222222
9,IBS25,4.0,80.953333,86.23,33.333333


Overall Average Final Exam Score and Average Final Score do follow a relatively expected trend, where students who performed better on the AP/IB exam also led to higher average scores.

In [11]:
# How do letter grades + counts differ between different groups?

letter_grade = joined_all.groupby(['Component', 'Score'])[["Final Grade"]].value_counts().reset_index().sort_values(["Component", "Score", "Final Grade"])
letter_grade

,Component,Score,Final Grade,count
2,AP20,1.0,A,1
3,AP20,1.0,A-,1
4,AP20,1.0,B,1
1,AP20,1.0,B+,2
0,AP20,1.0,C-,3
5,AP20,1.0,D,1
6,AP20,1.0,F,1
9,AP20,2.0,A,5
7,AP20,2.0,A-,8
8,AP20,2.0,B,8


### Grade Distribution by Biology Background

In [12]:
## Visualization: Bio 1B Final Grade Distribution by Biology Background

# map component codes to readable background labels
joined_all["bio_background"] = joined_all["Component"].map({
    "AP20": "AP Bio",
    "IBH10": "IB HL Bio",
    "IBS25": "IB SL Bio"
}).fillna("No Background")

# build counts first, then compute pct after concat
ap_ib = joined_all[joined_all["bio_background"] != "No Background"].groupby(["bio_background", "Score", "Final Grade"]).size().reset_index(name="count")

no_bg = joined_all[joined_all["bio_background"] == "No Background"].groupby("Final Grade").size().reset_index(name="count")
no_bg["bio_background"] = "No Background"
no_bg["Score"] = "None"

letter_pct = pd.concat([ap_ib, no_bg], ignore_index=True)

# compute pct after concat
letter_pct["pct"] = letter_pct["count"] / letter_pct.groupby(["bio_background", "Score"])["count"].transform("sum") * 100

# define grade order from worst to best so A+ stacks on top
grade_order = ["F", "D-", "D", "D+", "C-", "C", "C+", "B-", "B", "B+", "A-", "A", "A+"]

# deeper/richer colors for higher grades, lighter for lower grades
# A grades: deep green -> light green
# B grades: deep blue -> light blue
# C grades: deep yellow -> light yellow
# D grades: deep orange -> light orange
# F: red
colors = {
    "A+": "#145A32",  # deepest green
    "A":  "#1E8449",
    "A-": "#27AE60",
    "B+": "#1A5276",  # deepest blue
    "B":  "#2E86C1",
    "B-": "#7FB3D3",
    "C+": "#B7950B",  # deepest yellow
    "C":  "#D4AC0D",
    "C-": "#F4D03F",
    "D+": "#A04000",  # deepest orange
    "D":  "#E67E22",
    "D-": "#F0B27A",
    "F":  "#E74C3C",  # red
}

groups = ["AP Bio", "IB HL Bio", "IB SL Bio", "No Background"]
fig = make_subplots(rows=1, cols=4, subplot_titles=groups, shared_yaxes=True)

for i, group in enumerate(groups, 1):
    # filter to current group
    subset = letter_pct[letter_pct["bio_background"] == group]

    # pivot so rows = score, columns = letter grade
    pivot = subset.pivot_table(index="Score", columns="Final Grade", values="pct", aggfunc="sum")

    # reorder columns to match grade_order (only keep grades that exist in data)
    pivot = pivot.reindex(columns=[g for g in grade_order if g in pivot.columns])

    # add one bar trace per letter grade
    for grade in pivot.columns:
        fig.add_trace(
            go.Bar(
                name=grade,
                x=pivot.index.astype(str),
                y=pivot[grade],
                marker_color=colors[grade],
                # hover shows grade, score, and percentage
                hovertemplate=f"<b>{grade}</b><br>Score: %{{x}}<br>Percentage: %{{y:.1f}}%<extra></extra>",
                # only show legend for first subplot to avoid duplicates
                showlegend=(i == 1)
            ),
            row=1, col=i
        )

fig.update_layout(
    barmode="stack",
    title="Bio 1B Final Grade Distribution by Biology Background",
    height=500,
    legend_title="Final Grade",
)

# only label y axis on first subplot since they're shared
fig.update_yaxes(title_text="% of Students", row=1, col=1)

fig.show()

From this chart above, comparing just those that have taken AP Bio vs students with no background, those that received a 5 on the AP exam has almost double the amount of students receiving an A compared to students with no background, making up 65.6% and 31.9% of all students in the semester, respectively.

In [13]:
# Box plot: Final Score distribution by AP Bio background
group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
colors        = ['#9F77DD', '#9F77DD', '#9F77DD', '#9F77DD', '#9F77DD', '#7FB3D3']

def assign_group(row):
    if pd.isna(row['Score']):
        return 'No Background'
    elif row['Component'] == 'AP20':
        return f"{int(row['Score'])}"
    else:
        return 'Other IB'  # will be filtered out below

joined_all['Analysis_Group'] = joined_all.apply(assign_group, axis=1)

df_box = joined_all[joined_all['Analysis_Group'] != 'Other IB'].copy()

fig = go.Figure()
for group, display, color in zip(group_order, display_names, colors):
    vals = df_box[df_box['Analysis_Group'] == group]['Final Score'].dropna()
    fig.add_trace(go.Box(
        y=vals,
        name=display,
        marker_color=color,
        boxmean=True,  # shows mean as dashed line inside box
        hovertemplate=f"<b>{display}</b><br>Score: %{{y:.1f}}<extra></extra>"
    ))

fig.update_layout(
    title=dict(text='Final Grade Percentage Distribution by AP Biology Background', x=0.5),
    yaxis=dict(title='Final Score', gridcolor='#eeeeee'),
    xaxis=dict(title='Student Group'),
    plot_bgcolor='white',
    showlegend=False,
    height=450
)
fig.show()

In [14]:
joined_all.groupby("bio_background")[["Final Score", "Final Exam Final Score"]].agg(["mean", "count"]).round(2)

Final Score       Final Exam Final Score      
                      mean count                   mean count
bio_background                                               
AP Bio           90.412971   313              84.125272   313
IB HL Bio         88.98875    16              80.505625    16
IB SL Bio        86.697778     9              78.306667     9
No Background    87.768031   320                 80.246   320

If we focus on students who've taken AP Bio:

In [15]:
joined_all[joined_all["bio_background"] == "AP Bio"].groupby("Score")[["Final Score", "Final Exam Final Score"]].agg(["mean", "count"]).round(2)

Final Score       Final Exam Final Score      
             mean count                   mean count
Score                                               
1.0        78.704    10                 59.762    10
2.0     84.213514    37              76.641622    37
3.0     89.648902    82              82.985122    82
4.0       91.0295   100                84.4296   100
5.0     94.549524    84              91.072738    84

In this table, we can see that students who scored a 5 on the AP Bio exam has a higher average final exam score than students with no background. We also see that a small portion of students that scored a 1 on the AP Bio exam also received a lower average final exam score.

### Midterm Score Breakdown

In [16]:
# Define column groups (midterm exam scores)
midterm_mcq = [
    "Midterm 1 - Multiple Choice/Fixed Response Score (8916061)",
    "Midterm 2 -Multiple Choice/Fixed Response Score (8916064)",
    "Midterm 3 - Multiple Choice/Fixed Response (8916066)"
]
midterm_frq = [
    "Midterm 1 - Free Response Score (8916059)",
    "Midterm 2 - Free Response (8916062)",
    "Midterm 3 - Free Response (8916065)"
]

all_midterms = midterm_mcq + midterm_frq

# Create a mapping for easy percentage conversion
denominators = {col: 78 for col in midterm_mcq} | {col: 9 for col in midterm_frq}

# Isolate and calculate scores for "AP Bio" background (grouped by Score)
ap_bio_raw = joined_all[joined_all["bio_background"] == "AP Bio"]
ap_mt_scores = ap_bio_raw.groupby("Score")[all_midterms].mean()

# Convert AP Bio to percentages
ap_mt_scores = (ap_mt_scores / denominators * 100)

# Isolate and calculate scores for "No Background" (overall mean)
no_bg_raw = joined_all[joined_all["bio_background"] == "No Background"]
no_bg_mt_scores = no_bg_raw[all_midterms].mean().to_frame().T

# Convert No Background to percentages and fix index
no_bg_mt_scores = (no_bg_mt_scores / denominators * 100).round(2)
no_bg_mt_scores.index = ["No Background"]

# Stack them together and give the index a meaningful name
combined_scores = pd.concat([ap_mt_scores, no_bg_mt_scores])
combined_scores.index.name = "Student Group / Score"

# Display final table
combined_scores

,Midterm 1 - Multiple Choice/Fixed Response Score (8916061),Midterm 2 -Multiple Choice/Fixed Response Score (8916064),Midterm 3 - Multiple Choice/Fixed Response (8916066),Midterm 1 - Free Response Score (8916059),Midterm 2 - Free Response (8916062),Midterm 3 - Free Response (8916065)
Student Group / Score,,,,,,
1.0,72.000000,66.153846,58.846154,48.611111,42.222222,50.000000
2.0,75.567568,72.765073,65.696466,52.702703,58.408408,67.813814
3.0,83.073171,86.022514,76.031895,72.357724,78.658537,77.337398
4.0,88.400000,88.500000,79.346154,84.027778,87.625556,83.784444
5.0,91.476190,94.459707,86.172161,90.742063,90.542328,87.657407
No Background,82.610000,82.630000,72.190000,68.260000,73.600000,73.940000


In [17]:
# Grouped bar chart: Midterm scores by AP background
group_order   = [1.0, 2.0, 3.0, 4.0, 5.0, 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
colors        = ['#9F77DD', '#7FB3D3', '#5DCAA5', '#F4D03F', '#E67E22', '#95A5A6']

exam_pairs = [
    ('MCQ', "Midterm 1 - Multiple Choice/Fixed Response Score (8916061)", 78),
    ('MCQ', "Midterm 2 -Multiple Choice/Fixed Response Score (8916064)", 78),
    ('MCQ', "Midterm 3 - Multiple Choice/Fixed Response (8916066)", 78),
    ('FRQ', "Midterm 1 - Free Response Score (8916059)", 9),
    ('FRQ', "Midterm 2 - Free Response (8916062)", 9),
    ('FRQ', "Midterm 3 - Free Response (8916065)", 9),
]

midterm_labels = ['Midterm 1', 'Midterm 2', 'Midterm 3']

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{t} — {m}' for t in ['MCQ', 'FRQ'] for m in midterm_labels],
    shared_yaxes='rows',
    vertical_spacing=0.18,
    horizontal_spacing=0.06
)

for idx, (exam_type, col, denom) in enumerate(exam_pairs):
    row = 1 if exam_type == 'MCQ' else 2
    col_pos = (idx % 3) + 1

    for group, display, color in zip(group_order, display_names, colors):
        if group == 'No Background':
            val = no_bg_mt_scores[col].values[0]
            n   = len(no_bg_raw)
        else:
            if group not in ap_mt_scores.index:
                continue
            val = ap_mt_scores.loc[group, col]
            n   = len(ap_bio_raw[ap_bio_raw['Score'] == group])

        fig.add_trace(go.Bar(
            x=[display],
            y=[val],
            marker_color=color,
            name=display,
            showlegend=(idx == 0),
            legendgroup=display,
            text=f"{val:.1f}%",
            textposition='outside',
            hovertemplate=f"<b>{display}</b><br>Score: {val:.1f}%<br>n={n}<extra></extra>"
        ), row=row, col=col_pos)

fig.update_layout(
    title=dict(text='Midterm Score Breakdown by AP Biology Background', x=0.5),
    barmode='group',
    height=600,
    plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
)

for r in [1, 2]:
    fig.update_yaxes(range=[0, 115], ticksuffix='%', gridcolor='#eeeeee', row=r, col=1)

fig.show()

Looking at the Midterm score breakdown, we can see a positive correlation between prior AP Biology knowledge and exam performance. Students who earned a score of 4 or 5 on the AP Biology exam consistently outperformed peers with no prior biology background across all three midterms. This performance gap is most noticeable in the Free Response Questions. For instance, students with an AP score of 5 averaged roughly 90% across all FRQ sections, whereas students with no background averaged between 68% and 74%.

#### We can test for the differences between these groups using Analysis of Variance (ANOVA) to measure the amount of variance between different score groups to the amount of variance within the group

## Step 1 — What is ANOVA and why are we using it?

**ANOVA (Analysis of Variance)** answers one question:
> *Are the average scores of these groups different enough that it's unlikely to be random chance?*

In our case, the groups are students' AP Bio scores: **1, 2, 3, 4, 5, and No Background**.

**Why not just compare means?**  
We could eyeball the averages, but that doesn't tell us if the differences are *statistically meaningful* or just noise. ANOVA gives us a **p-value** — the probability that differences this large would appear even if all groups were actually equal. A p-value below 0.05 means: *"less than 5% chance this is a fluke."*

**Limitation:** ANOVA only says *"at least one group is different"*. It doesn't say *which* groups differ. That's what the **Tukey HSD** test (Step 3) handles.

### Step 2 — Build the analysis groups and clean up makeup exam scores

In [18]:
# ── 2a. Label each student with their AP Bio score group ──────────────────────
# We only keep AP Bio students (AP20) and students with no biology background.
# IB students are excluded so we have a clean, interpretable comparison.

def assign_group(row):
    if pd.isna(row['Score']):
        return 'No Background'
    elif row['Component'] == 'AP20':
        return f"{int(row['Score'])}"
    else:
        return 'Other IB'  # will be filtered out below

joined_all['Analysis_Group'] = joined_all.apply(assign_group, axis=1)

print("Group counts (before filtering IB):")
print(joined_all['Analysis_Group'].value_counts())


Group counts (before filtering IB):
Analysis_Group
No Background    320
4                100
5                 84
3                 82
2                 37
Other IB          25
1                 10
Name: count, dtype: int64


In [19]:
# ── 2b. Exam scores: use raw score, but fill in makeup ONLY if raw = 0 ────────
# 0 means the student missed the original exam entirely.
# In that case we substitute their makeup score so they aren't excluded.
# Students who took the original exam keep their score regardless of makeup.

def get_score(raw_col, makeup_col=None):
    raw = joined_all[raw_col]
    if makeup_col is None:
        return raw  # no makeup exists for this exam
    return raw.replace(0, np.nan).fillna(joined_all[makeup_col]).fillna(0)

EXAMS = {
    'MT1_MCQ': 'Midterm 1 MCQ',
    'MT1_FRQ': 'Midterm 1 FRQ',
    'MT2_MCQ': 'Midterm 2 MCQ',
    'MT2_FRQ': 'Midterm 2 FRQ',
    'MT3_MCQ': 'Midterm 3 MCQ',
    'MT3_FRQ': 'Midterm 3 FRQ',
}

joined_all['MT1_MCQ'] = get_score('Midterm 1 - Multiple Choice/Fixed Response Score (8916061)',
                                   'Midterm 1 - MC/fixed Response Make up (8916060)')
joined_all['MT1_FRQ'] = get_score('Midterm 1 - Free Response Score (8916059)')   # no makeup column

joined_all['MT2_MCQ'] = get_score('Midterm 2 -Multiple Choice/Fixed Response Score (8916064)',
                                   'Midterm 2 - MC/Fixed Response Make up (8916063)')
joined_all['MT2_FRQ'] = get_score('Midterm 2 - Free Response (8916062)')          # no makeup column

joined_all['MT3_MCQ'] = get_score('Midterm 3 - Multiple Choice/Fixed Response (8916066)',
                                   'Midterm 3 - Multiple Choice/Fixed Response Make Up (8916067)')
joined_all['MT3_FRQ'] = get_score('Midterm 3 - Free Response (8916065)')          # no makeup column

print("Scores ready:", list(EXAMS.keys()))

Scores ready: ['MT1_MCQ', 'MT1_FRQ', 'MT2_MCQ', 'MT2_FRQ', 'MT3_MCQ', 'MT3_FRQ']


### Step 3 — Run the one-way ANOVA

For each exam we:
1. Pull out only AP Bio and No Background students
2. Fit a simple linear model: `score ~ group`
3. Run ANOVA on that model to get an F-statistic and p-value

**Reading the output:**
- `PR(>F)` is the p-value — the star column on the right makes it easy to spot significant results
- `< 0.05` → significant (*)
- `< 0.01` → very significant (**)  
- `< 0.001` → highly significant (***)

In [20]:
print("=" * 65)
print(f"{'Exam':<22} {'F-stat':>10} {'p-value':>14}  {'Sig':>4}")
print("=" * 65)

for col, label in EXAMS.items():
    # Keep only AP Bio + No Background students, drop any missing scores
    df = (joined_all[['Analysis_Group', col]]
          .query("Analysis_Group != 'Other IB'")
          .dropna())
    df.columns = ['Group', 'Score']

    # Fit model: Score is explained by which Group the student is in
    model = ols('Score ~ C(Group)', data=df).fit()

    # anova_lm extracts the ANOVA table from that model
    anova_table = sm.stats.anova_lm(model, typ=2)

    f_stat = anova_table.loc['C(Group)', 'F']
    p_val  = anova_table.loc['C(Group)', 'PR(>F)']

    # Significance stars for quick reading
    if   p_val < 0.001: sig = '***'
    elif p_val < 0.01:  sig = '** '
    elif p_val < 0.05:  sig = '*  '
    else:               sig = '   '

    print(f"{label:<22} {f_stat:>10.3f} {p_val:>14.8f}  {sig}")

print("=" * 65)
print("Significance: * p<0.05   ** p<0.01   *** p<0.001")


Exam                       F-stat        p-value   Sig
Midterm 1 MCQ              13.510     0.00000000  ***
Midterm 1 FRQ              23.340     0.00000000  ***
Midterm 2 MCQ              18.410     0.00000000  ***
Midterm 2 FRQ              22.699     0.00000000  ***
Midterm 3 MCQ              14.718     0.00000000  ***
Midterm 3 FRQ              14.163     0.00000000  ***
Significance: * p<0.05   ** p<0.01   *** p<0.001


Since the p-value is all less than 0.01, we can state that the differences in scores between AP Bio groups and No Background students are so large that there is less than a 0.1% chance they happened by random chance.

The F-stat explains the strength of the group effect — higher = the groups are more spread apart relative to the noise within each group. The FRQ exams (Midterm 1: 23.3 and Midterm 2: 22.7) actually have a stronger group effect than MCQ, which means the groups are more cleanly separated on FRQ relative to the noise within each group.

### Step 4 — Tukey HSD post-hoc test

ANOVA told us *something* is different — now Tukey HSD tells us *which pairs*.

It compares every possible pair of groups (e.g. Score 5 vs No Background, Score 5 vs Score 1, etc.) and gives each pair:
- **meandiff** — how many raw points apart the two groups are on average
- **p-adj** — p-value adjusted for multiple comparisons (so we don't get false positives from testing so many pairs)
- **reject** — `True` means the difference IS statistically significant

In [21]:
# ── 1. GLOBAL CONFIGURATION & VARIABLE MAPPING ───────────────────────────────
exams = {
    'MT1_MCQ': 'Midterm 1 MCQ',
    'MT1_FRQ': 'Midterm 1 FRQ',
    'MT2_MCQ': 'Midterm 2 MCQ',
    'MT2_FRQ': 'Midterm 2 FRQ',
    'MT3_MCQ': 'Midterm 3 MCQ',
    'MT3_FRQ': 'Midterm 3 FRQ',
}

group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
name_mapping  = dict(zip(group_order, display_names))


# ── 2. PROCESS DATA AND LOOP GENERATE PLOTS ──────────────────────────────────
for col, label in exams.items():

    # Clean data specifically for the current exam column
    df_t = (joined_all[['Analysis_Group', col]]
            .query("Analysis_Group != 'Other IB'")
            .dropna())
    df_t.columns = ['Group', 'Score']

    # Calculate Tukey HSD
    tukey = pairwise_tukeyhsd(endog=df_t['Score'], groups=df_t['Group'], alpha=0.05)

    # Extract the pooled standard deviation using the Tukey test variance estimation
    # (Tukey's standard error = pooled_sd * sqrt(1/n1 + 1/n2))
    # Standard math logic: pooled within-group variance can be backed out from statsmodels' internal attributes
    pooled_sd = np.sqrt(tukey.variance)

    # Format the statsmodels Tukey summary matrix into a Pandas DataFrame
    data = tukey.summary().data[1:]
    columns = tukey.summary().data[0]
    df_res = pd.DataFrame(data, columns=columns)

    # Clean and parse types
    df_res['group1']   = df_res['group1'].astype(str).map(name_mapping)
    df_res['group2']   = df_res['group2'].astype(str).map(name_mapping)
    df_res['meandiff'] = df_res['meandiff'].astype(float)
    df_res['lower']    = df_res['lower'].astype(float)
    df_res['upper']    = df_res['upper'].astype(float)
    df_res['reject']   = df_res['reject'].astype(bool)
    df_res['p-adj']    = df_res['p-adj'].astype(float)
    df_res['comparison'] = df_res['group1'] + ' vs ' + df_res['group2']

    # STANDARDIZATION STEP: Convert raw point metrics into Cohen's d scale units
    df_res['cohens_d']       = df_res['meandiff'] / pooled_sd
    df_res['cohens_d_lower'] = df_res['lower'] / pooled_sd
    df_res['cohens_d_upper'] = df_res['upper'] / pooled_sd

    # ── 3. BUILD THE STANDARDIZED COHEN'S D PLOT ─────────────────────────────
    fig = go.Figure()

    plot_colors = ['#E74C3C' if r else '#7F8C8D' for r in df_res['reject']]

    for idx, row in df_res.iterrows():
        fig.add_trace(go.Scatter(
            x=[row['cohens_d_lower'], row['cohens_d_upper']],
            y=[row['comparison'], row['comparison']],
            mode='lines+markers',
            marker=dict(size=8, color=plot_colors[idx]),
            line=dict(width=3, color=plot_colors[idx]),
            hovertemplate=(
                f"<b>{row['comparison']}</b><br>"
                f"Cohen's d: {row['cohens_d']:.2f}<br>"
                f"95% CI: [{row['cohens_d_lower']:.2f}, {row['cohens_d_upper']:.2f}]<br>"
                f"p-adj: {row['p-adj']:.4f}<extra></extra>"
            )
        ))

    # Draw vertical reference marker line at 0
    fig.add_vline(x=0, line_width=2, line_dash="dash", line_color="#333333")

    # Apply clean chart layouts with standardized X-axis labels
    fig.update_layout(
        title=dict(
            text=f"Standardized Tukey HSD Effect Sizes (Cohen's d) — {label}<br>"
                 f"<sup>Values represent differences in Standard Deviation units</sup>",
            x=0.5
        ),
        xaxis=dict(
            title="Effect Size (Cohen's d)",
            gridcolor='#eeeeee',
            # Fixed range if you want all charts on the exact same scale, e.g., range=[-2.5, 2.5]
        ),
        yaxis=dict(title='', categoryorder='trace'),
        plot_bgcolor='white',
        showlegend=False,
        height=200 + (len(df_res) * 30)
    )

    fig.show()

## Tukey HSD Post-Hoc Analysis (Standardized Effect Sizes)

Following a significant one-way ANOVA, a Tukey HSD post-hoc test was conducted to identify specific pairwise differences across all six midterm components. To allow for direct comparison between Multiple Choice (MCQ) and Free Response (FRQ) formats, mean differences were standardized into **Cohen’s $d$ effect sizes** using the pooled within-group standard deviation.

Groups are categorized by their AP Biology exam score (1–5) and a "No Background" reference group (students with no prior AP or IB Biology coursework). IB students were excluded to maintain a clean comparison.

### Midterm 1

#### Multiple Choice (MCQ)
* **Significant Advantages over No Background:** Students with strong AP preparation showed a substantial, large advantage over those with no biology background (**AP 5**: $d$ = 0.93, $p$ < 0.001; **AP 4**: $d$ = 0.49, $p$ = 0.009). A moderate significant separation also appeared for **AP 2** ($d$ = 0.62, $p$ = 0.048).
* **Other Significant Separations:** **AP 5** students established an exceptionally large margin over lower-scoring AP tiers, significantly outperforming both **AP 1** ($d$ = 1.86, $p$ < 0.001) and **AP 2** ($d$ = 1.55, $p$ < 0.001).
* **Muted/Non-Significant Pairs:** Horizontal confidence lines cross 0 for **AP 1** ($p$ = 0.999) and **AP 3** ($p$ = 0.992) when compared to the **No Background** cohort, showing no statistically detectable difference.

#### Free Response (FRQ)
* **Significant Advantages over No Background:** A strong performance gap emerged on the written section. **AP 5** ($d$ = 1.05, $p$ < 0.001) and **AP 4** ($d$ = 0.74, $p$ < 0.001) cohorts exhibited large effect size advantages over the **No Background** baseline. **AP 2** also outpaced **No Background** ($d$ = 0.73, $p$ = 0.003).
* **Other Significant Separations:** Reflecting weak foundational performance, the **AP 1** group scored significantly lower than the **AP 3**, **AP 4**, and **AP 5** cohorts.
* **Muted/Non-Significant Pairs:** The confidence interval for **AP 3 vs. No Background** widely straddles zero, demonstrating no statistical benefit ($d$ = 0.17, $p$ = 0.757).

### Midterm 2

#### Multiple Choice (MCQ)
* **Significant Advantages over No Background:** The separation widened considerably, with **AP 5** students demonstrating a massive advantage over **No Background** ($d$ = 1.02, $p$ < 0.001).
* **Other Significant Separations:** **AP 5** students established clear dominance, significantly outperforming all other cohorts, including **AP 4** ($d$ = 0.75, $p$ = 0.003). Conversely, **AP 1** students severely underperformed relative to **AP 3**, **AP 4**, **AP 5**, and **No Background** ($d$ = -1.47, $p$ = 0.006).
* **Muted/Non-Significant Pairs:** **AP 3** performance remained statistically indistinguishable from **No Background** with a negligible effect size ($d$ = 0.17, $p$ = 0.543).

#### Free Response (FRQ)
* **Significant Advantages over No Background:** **AP 4** ($d$ = 0.74, $p$ < 0.001) and **AP 5** ($d$ = 0.89, $p$ < 0.001) intervals sat entirely to the right of zero, representing large, statistically robust advantages over the **No Background** group.
* **Other Significant Separations:** **AP 1** students continued to show severe deficits, scoring significantly lower than every other student cohort except **AP 2**.
* **Muted/Non-Significant Pairs:** The interval line for **AP 3 vs. No Background** comfortably crossed zero, showing no meaningful difference ($d$ = 0.22, $p$ = 0.392).

### Midterm 3

#### Multiple Choice (MCQ)
* **Significant Advantages over No Background:** High-scoring AP cohorts demonstrated their most pronounced visual separation on this chart. **AP 5** students outpaced **No Background** by an enormous margin ($d$ = 1.18, $p$ < 0.001), marking the largest positive effect size observed in the study. **AP 4** also maintained a large, significant advantage ($d$ = 0.59, $p$ = 0.003).
* **Muted/Non-Significant Pairs:** **AP 3** stayed statistically flat against **No Background** with no real variance ($d$ = 0.08, $p$ = 0.908). The interval for **AP 1 vs. No Background** just barely clipped the zero threshold line, approaching but missing statistical significance ($d$ = 1.28, $p$ = 0.051).

#### Free Response (FRQ)
* **Significant Advantages over No Background:** Clear, positive separation from **No Background** was maintained by both **AP 4** ($d$ = 0.61, $p$ < 0.001) and **AP 5** ($d$ = 0.84, $p$ < 0.001), showing large effect sizes on the application-heavy written material.
* **Other Significant Separations:** **AP 1** students scored significantly lower than **AP 3**, **AP 4**, **AP 5**, and **No Background** cohorts ($d$ = -1.47, $p$ = 0.002).
* **Muted/Non-Significant Pairs:** The **AP 3 vs. No Background** interval continued to sit directly over zero ($d$ = 0.14, $p$ = 0.669).

---

### Summary

Across all six midterm evaluations, standardizing the data into Cohen’s $d$ effect sizes exposes an unambiguous structural divergence in student outcomes:

1. **The High-Performance Tier:** Students entering with an **AP 5 or AP 4** template consistently outpace peers with no background. This advantage is mathematically large ($d$ > 0.80) and steady across the semester, peaking on the Midterm 3 MCQ section ($d$ = 1.18).
2. **The Baseline Tier:** Students entering with an **AP 3 or lower** show no statistically verifiable advantage over the **No Background** cohort. Their effect sizes are uniformly small to negligible ($d$ $\approx$ 0.10), sitting comfortably inside the non-significant gray zones of the charts.
3. **The Compounding Effect:** The systematic, severe underperformance of AP 1 students ($d$ ranges from -1.40 to -1.80 against higher tiers) alongside the expanding gains of AP 5 students highlights that the advantages of AP preparation compound over time as the coursework moves into advanced material.

## Score 3 vs No Background Comparison
Investigating whether AP Bio score 3 students perform significantly differently from students with no biology background.

In [22]:
# compare score 3 vs no background directly
comparison = joined_all[
    ((joined_all["bio_background"] == "AP Bio") & (joined_all["Score"] == 3.0)) |
    (joined_all["bio_background"] == "No Background")
].groupby("bio_background")[["Final Score", "Final Exam Final Score"]].agg(["mean", "std", "count"]).round(2)

comparison

Final Score             Final Exam Final Score             
                      mean   std count                   mean    std count
bio_background                                                            
AP Bio           89.648902  6.01    82              82.985122  12.76    82
No Background    87.768031  9.30   320                 80.246  14.14   320

In [23]:
joined_all["Final Score"] = pd.to_numeric(joined_all["Final Score"], errors="coerce")

score3 = joined_all[(joined_all["bio_background"] == "AP Bio") & (joined_all["Score"] == 3.0)]["Final Score"].dropna()
no_bg = joined_all[joined_all["bio_background"] == "No Background"]["Final Score"].dropna()

t_stat, p_val = stats.ttest_ind(score3, no_bg)
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_val:.4f}")
print(f"Score 3 mean: {score3.mean():.2f}, No Background mean: {no_bg.mean():.2f}")

t-statistic: 1.740
p-value: 0.0826
Score 3 mean: 89.65, No Background mean: 87.77


Students who scored a 3 on AP Biology perform similarly to students with no biology background at all (89.7 vs 87.8 mean final score, p = 0.079). This suggests that a passing AP Bio score of 3 does not confer a meaningful academic advantage in Bio 1B. The performance benefit becomes significant only at scores 4 and 5.

In [24]:
# score 2 vs no background
score2 = joined_all[(joined_all["bio_background"] == "AP Bio") & (joined_all["Score"] == 2.0)]["Final Score"].dropna()
t_stat2, p_val2 = stats.ttest_ind(score2, no_bg)
print(f"=== Score 2 vs No Background ===")
print(f"t-statistic: {t_stat2:.3f}")
print(f"p-value: {p_val2:.4f}")
print(f"Score 2 mean: {score2.mean():.2f}, No Background mean: {no_bg.mean():.2f}")

print()

# score 4 vs no background
score4 = joined_all[(joined_all["bio_background"] == "AP Bio") & (joined_all["Score"] == 4.0)]["Final Score"].dropna()
t_stat4, p_val4 = stats.ttest_ind(score4, no_bg)
print(f"=== Score 4 vs No Background ===")
print(f"t-statistic: {t_stat4:.3f}")
print(f"p-value: {p_val4:.4f}")
print(f"Score 4 mean: {score4.mean():.2f}, No Background mean: {no_bg.mean():.2f}")

print()

# score 5 vs no background
score5 = joined_all[(joined_all["bio_background"] == "AP Bio") & (joined_all["Score"] == 5.0)]["Final Score"].dropna()
t_stat5, p_val5 = stats.ttest_ind(score5, no_bg)
print(f"=== Score 5 vs No Background ===")
print(f"t-statistic: {t_stat5:.3f}")
print(f"p-value: {p_val5:.4f}")
print(f"Score 5 mean: {score5.mean():.2f}, No Background mean: {no_bg.mean():.2f}")

=== Score 2 vs No Background ===
t-statistic: -2.152
p-value: 0.0321
Score 2 mean: 84.21, No Background mean: 87.77

=== Score 4 vs No Background ===
t-statistic: 3.034
p-value: 0.0026
Score 4 mean: 91.03, No Background mean: 87.77

=== Score 5 vs No Background ===
t-statistic: 6.567
p-value: 0.0000
Score 5 mean: 94.55, No Background mean: 87.77


### Midterm Scores Over Time by AP Background

In [25]:
# Denominators
MCQ_MAX = 78
FRQ_MAX = 9

# Groups
groups        = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
colors        = ['#9F77DD', '#5DCAA5', '#F4D03F', '#E67E22', '#E74C3C', '#7FB3D3']
midterms      = ['Midterm 1', 'Midterm 2', 'Midterm 3']

# Compute means
df_plot = joined_all[joined_all['Analysis_Group'] != 'Other IB'].copy()

mcq_cols = ['MT1_MCQ', 'MT2_MCQ', 'MT3_MCQ']
frq_cols = ['MT1_FRQ', 'MT2_FRQ', 'MT3_FRQ']

mcq_means = (df_plot.groupby('Analysis_Group')[mcq_cols].mean() / MCQ_MAX * 100)
frq_means = (df_plot.groupby('Analysis_Group')[frq_cols].mean() / FRQ_MAX * 100)
mcq_means.columns = midterms
frq_means.columns = midterms

# also compute sample sizes for hover
n_per_group = df_plot.groupby('Analysis_Group').size()

# shared y range
all_vals = pd.concat([mcq_means, frq_means]).values.flatten()
y_min = max(0,  all_vals.min() - 5)
y_max = min(100, all_vals.max() + 5)

# Build figure
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MCQ (% of 78 pts)', 'FRQ (% of 9 pts)'],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for col_idx, (means, exam_label) in enumerate([(mcq_means, 'MCQ'), (frq_means, 'FRQ')], start=1):
    for group, display, color in zip(groups, display_names, colors):
        if group not in means.index:
            continue

        vals = means.loc[group]
        n    = n_per_group.get(group, 0)

        # hover text per point
        hover = [
            f"<b>{display}</b><br>"
            f"{mt}: {v:.1f}%<br>"
            f"n = {n} students"
            for mt, v in zip(midterms, vals)
        ]

        fig.add_trace(
            go.Scatter(
                x=midterms,
                y=vals.values,
                mode='lines+markers',
                name=display,
                line=dict(
                    color=color,
                    width=3 if group in ('5', 'No Background') else 1.8,
                    dash='dash' if group == 'No Background' else 'solid'
                ),
                marker=dict(size=9 if group in ('5', 'No Background') else 7),
                hovertemplate='%{customdata}<extra></extra>',
                customdata=hover,
                legendgroup=display,        # link MCQ + FRQ traces in legend
                showlegend=(col_idx == 1),  # only show legend entry once
            ),
            row=1, col=col_idx
        )

# Layout
fig.update_layout(
    title=dict(
        text='Mean Midterm Score Over Time by AP Biology Background',
        font=dict(size=15),
        x=0.5
    ),
    yaxis=dict(range=[y_min, y_max], ticksuffix='%', title='Mean Score (%)'),
    yaxis2=dict(range=[y_min, y_max], ticksuffix='%'),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5,
        font=dict(size=11)
    ),
    hovermode='x unified',
    plot_bgcolor='white',
    width=900,
    height=480,
)

# Clean grid styling
for axis in ['xaxis', 'xaxis2']:
    fig.update_layout({axis: dict(showgrid=False, linecolor='lightgrey')})
for axis in ['yaxis', 'yaxis2']:
    fig.update_layout({axis: dict(gridcolor='#eeeeee', gridwidth=1)})

fig.show()

In [26]:
# MCQ vs FRQ breakdown for no background students
no_bg_mt = joined_all[joined_all["bio_background"] == "No Background"][[
    "Midterm 1 - Multiple Choice/Fixed Response Score (8916061)",
    "Midterm 1 - Free Response Score (8916059)",
    "Midterm 2 -Multiple Choice/Fixed Response Score (8916064)",
    "Midterm 2 - Free Response (8916062)",
    "Midterm 3 - Multiple Choice/Fixed Response (8916066)",
    "Midterm 3 - Free Response (8916065)"
]].mean().round(2).to_frame().T

# normalize to % correct
no_bg_mt[midterm_mcq] = no_bg_mt[midterm_mcq] / 78 * 100
no_bg_mt[midterm_frq] = no_bg_mt[midterm_frq] / 9 * 100
no_bg_mt.index = ["No Background"]

# put side by side with AP Bio breakdown
comparison = pd.concat([ap_mt_scores, no_bg_mt])
comparison

,Midterm 1 - Multiple Choice/Fixed Response Score (8916061),Midterm 2 -Multiple Choice/Fixed Response Score (8916064),Midterm 3 - Multiple Choice/Fixed Response (8916066),Midterm 1 - Free Response Score (8916059),Midterm 2 - Free Response (8916062),Midterm 3 - Free Response (8916065)
1.0,72.000000,66.153846,58.846154,48.611111,42.222222,50.000000
2.0,75.567568,72.765073,65.696466,52.702703,58.408408,67.813814
3.0,83.073171,86.022514,76.031895,72.357724,78.658537,77.337398
4.0,88.400000,88.500000,79.346154,84.027778,87.625556,83.784444
5.0,91.476190,94.459707,86.172161,90.742063,90.542328,87.657407
No Background,82.615385,82.628205,72.192308,68.222222,73.555556,73.888889


## AI Tool Usage by AP Background

## Section 1: AI Tool Usage Overview

**Survey question:** Q133 — "Did you use AI tools to help you in
this course?" (Yes / No)

**Survey response rate:** 402 out of 658 enrolled students answered
this question. All students are preserved via a left join — students
who did not respond are excluded only from AI-specific analyses.

In [27]:
# Cross-tab: AI usage by AP background group
df_ai = joined_all[joined_all['Analysis_Group'] != 'Other IB'].copy()
df_ai = df_ai.merge(df_survey[['UID', 'Q133']], on='UID', how='left')

# Count and compute percentages
ct = pd.crosstab(df_ai['Analysis_Group'], df_ai['Q133'])
ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)

group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
n_labels      = [str(ct.loc[g].sum()) if g in ct.index else '0' for g in group_order]

fig = go.Figure()

# Yes bar
yes_pcts = [ct_pct.loc[g, 'Yes'] if g in ct_pct.index and 'Yes' in ct_pct.columns else 0
            for g in group_order]
no_pcts  = [ct_pct.loc[g, 'No']  if g in ct_pct.index and 'No'  in ct_pct.columns else 0
            for g in group_order]
yes_ns   = [ct.loc[g, 'Yes'] if g in ct.index and 'Yes' in ct.columns else 0 for g in group_order]
no_ns    = [ct.loc[g, 'No']  if g in ct.index and 'No'  in ct.columns else 0 for g in group_order]

fig.add_trace(go.Bar(
    name='Uses AI',
    x=display_names,
    y=yes_pcts,
    marker_color='#5DCAA5',
    text=[f'{p:.0f}%<br>(n={n})' for p, n in zip(yes_pcts, yes_ns)],
    textposition='inside',
    hovertemplate='<b>%{x}</b><br>Uses AI: %{y:.1f}%<extra></extra>'
))

fig.add_trace(go.Bar(
    name='Does Not Use AI',
    x=display_names,
    y=no_pcts,
    marker_color='#E8E8E8',
    text=[f'{p:.0f}%<br>(n={n})' for p, n in zip(no_pcts, no_ns)],
    textposition='inside',
    textfont=dict(color='#666666'),
    hovertemplate='<b>%{x}</b><br>Does not use AI: %{y:.1f}%<extra></extra>'
))

fig.update_layout(
    barmode='stack',
    title=dict(text='AI Tool Usage by AP Biology Background', x=0.5, font=dict(size=15)),
    xaxis=dict(title='Student Group'),
    yaxis=dict(title='% of Students', ticksuffix='%', range=[0, 110]),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    plot_bgcolor='white',
    height=450,
    annotations=[
        dict(x=name, y=103, text=f'n={n}', showarrow=False,
             font=dict(size=11, color='#444444'), xref='x', yref='y')
        for name, n in zip(display_names, n_labels)
    ]
)
fig.update_yaxes(gridcolor='#eeeeee')
fig.show()

## Section 2: Does AI Usage Predict Final Exam Performance Overall?

We use the **Final Exam Final Score** as
our outcome variable.

A **Welch's t-test** compares mean final exam scores between AI
users and non-users. Welch's variant is used instead of the standard
t-test because it does not assume equal variance across groups.


In [28]:
# merge AI usage (Q133) from survey onto joined_all
# left join so all students are preserved even if they didn't complete survey
df_ai = joined_all[joined_all['Analysis_Group'] != 'Other IB'].copy()
df_ai = df_ai.merge(df_survey[['UID', 'Q133']], on='UID', how='left')

# check how many students answered the survey question
print(df_ai['Q133'].value_counts(dropna=False))
print(f"\nSurvey response rate: {df_ai['Q133'].notna().sum()} / {len(df_ai)} students")

Q133
No     241
NaN    230
Yes    162
Name: count, dtype: int64

Survey response rate: 403 / 633 students


In [29]:
# drop students who didn't answer the survey question
# force numeric and drop missing

df_ai_clean = df_ai.dropna(subset=['Q133']).copy()
df_ai_clean["Final Exam Final Score"] = pd.to_numeric(df_ai_clean["Final Exam Final Score"], errors="coerce")
df_ai_clean = df_ai_clean.dropna(subset=["Final Exam Final Score"])

ai_yes = df_ai_clean[df_ai_clean['Q133'] == 'Yes']['Final Exam Final Score']
ai_no  = df_ai_clean[df_ai_clean['Q133'] == 'No']['Final Exam Final Score']

# Welch's t-test (doesn't assume equal variances)
t_stat, p_val = stats.ttest_ind(ai_yes, ai_no, equal_var=False)

# Cohen's d effect size
pooled_sd = np.sqrt(
    ((len(ai_yes)-1)*ai_yes.var() + (len(ai_no)-1)*ai_no.var()) /
    (len(ai_yes) + len(ai_no) - 2)
)
cohens_d = (ai_yes.mean() - ai_no.mean()) / pooled_sd

print(f"Uses AI:   mean={ai_yes.mean():.2f}, n={len(ai_yes)}")
print(f"No AI:     mean={ai_no.mean():.2f}, n={len(ai_no)}")
print(f"Difference: {ai_yes.mean() - ai_no.mean():.2f} points")
print(f"Welch's t-test: t={t_stat:.3f}, p={p_val:.4f}")
print(f"Cohen's d: {cohens_d:.3f}")

Uses AI:   mean=82.38, n=162
No AI:     mean=83.27, n=241
Difference: -0.90 points
Welch's t-test: t=-0.687, p=0.4928
Cohen's d: -0.070


In [30]:
group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']

results = []
for group, display in zip(group_order, display_names):
    subset = df_ai_clean[df_ai_clean['Analysis_Group'] == group]
    ai_yes = subset[subset['Q133'] == 'Yes']['Final Exam Final Score']
    ai_no  = subset[subset['Q133'] == 'No']['Final Exam Final Score']

    if len(ai_yes) < 3 or len(ai_no) < 3:
        results.append({'Group': display, 'AI Mean': None, 'No AI Mean': None,
                        'Difference': None, 'n (AI)': len(ai_yes),
                        'n (No AI)': len(ai_no), 'p-value': None, 'Significant': None})
        continue

    t_stat, p_val = stats.ttest_ind(ai_yes, ai_no, equal_var=False)
    pooled_sd = np.sqrt(
        ((len(ai_yes)-1)*ai_yes.var() + (len(ai_no)-1)*ai_no.var()) /
        (len(ai_yes) + len(ai_no) - 2)
    )
    d = (ai_yes.mean() - ai_no.mean()) / pooled_sd

    results.append({
        'Group': display,
        'AI Mean': round(ai_yes.mean(), 2),
        'No AI Mean': round(ai_no.mean(), 2),
        'Difference': round(ai_yes.mean() - ai_no.mean(), 2),
        'n (AI)': len(ai_yes),
        'n (No AI)': len(ai_no),
        'p-value': round(p_val, 4),
        "Cohen's d": round(d, 3),
        'Significant': p_val < 0.05
    })

df_results = pd.DataFrame(results)
df_results

,Group,AI Mean,No AI Mean,Difference,n (AI),n (No AI),p-value,Significant,Cohen's d
0,AP 1,NaN,NaN,NaN,0,5,NaN,None,NaN
1,AP 2,80.95,77.78,3.18,10,15,0.5147,False,0.277
2,AP 3,83.59,84.24,-0.65,28,29,0.8646,False,-0.046
3,AP 4,83.90,85.50,-1.59,25,44,0.4385,False,-0.196
4,AP 5,88.61,90.99,-2.38,14,28,0.2950,False,-0.376
5,No Background,80.67,81.85,-1.17,85,120,0.5404,False,-0.084


### Interpretation
Using the final exam score, AI tool usage shows no statistically
significant association with performance overall (t=-0.685, p=0.49)
or within any AP score group.

**Limitation:** Only 402 of 658 students completed the survey
question, which reduces statistical power especially in smaller
AP score groups. A larger or more complete survey response rate
would be needed to draw firm conclusions about AI usage effects
within subgroups.

## Section 3: Does AI Usage Differ by AP Background Group?

Breaking down by AP score group reveals a more nuanced picture,
though no individual group reaches statistical significance —
likely due to small subgroup sample sizes.

**Key observations:**
- **AP 2 students** show the largest positive gap: AI users scored
  3.18 points higher on the final exam (80.95 vs 77.78, d=0.277).
  Not significant (p=0.51) but the moderate effect size is
  directionally meaningful given small n=10/15.
- **AP 5 students** show the opposite trend: AI users scored 2.38
  points *lower* (88.61 vs 90.99, d=-0.376). Suggests highly
  prepared students do not benefit from — and may be slightly
  disadvantaged by — AI tool usage.
- **No Background, AP 3, AP 4** all show negligible differences
  close to zero.

**Hypothesis:** AI tools may provide modest benefit for
underprepared students (AP 2) but offer no advantage for highly
prepared students (AP 5) who may not need supplemental support.

In [31]:
# 1. Clean data using your active notebook columns
df_breakdown = (df_ai.dropna(subset=['Q133', 'Final Exam Final Score', 'Analysis_Group'])
                .query("Analysis_Group != 'Other IB'"))

# 2. Consistent ordering and mapping from your previous notebook cells
group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
name_mapping  = dict(zip(group_order, display_names))

# Map raw group values ('1', '2', etc.) to your clean display names ('AP 1', etc.)
df_breakdown['AP_Group'] = df_breakdown['Analysis_Group'].astype(str).map(name_mapping)

fig = go.Figure()

# 3. Separate categories by AI tool usage (Q133)
for usage, color in [('Yes', '#5DCAA5'), ('No', '#7FB3D3')]:
    display_name = 'Uses AI' if usage == 'Yes' else 'Does Not Use AI'
    df_sub = df_breakdown[df_breakdown['Q133'] == usage]

    x_labels = []
    y_vals = []

    # Align values matching your precise group order sequence
    for ap_label in display_names:
        scores = df_sub[df_sub['AP_Group'] == ap_label]['Final Exam Final Score']

        if not scores.empty:
            x_labels.extend([ap_label] * len(scores))
            y_vals.extend(scores.tolist())

            # Calculate the group subset mean score
            g_mean = scores.mean()

            # Position the text annotation directly over the specific category column
            fig.add_annotation(
                x=ap_label,
                y=scores.max() + 3.0,
                text=f"<b>{g_mean:.1f}</b>",
                showarrow=False,
                font=dict(size=10, color='#2c3e50'),
                # Shunts text left or right to align perfectly with grouped bars
                xshift=-45 if usage == 'Yes' else 45,
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='#e0e0e0',
                borderpad=2
            )

    # Add the grouped data columns
    fig.add_trace(go.Box(
        x=x_labels,
        y=y_vals,
        name=display_name,
        marker_color=color,
        boxmean=True,
        offsetgroup=display_name,
        hovertemplate=f"<b>{display_name}</b> — %{{x}}<br>Score: %{{y:.1f}}<extra></extra>"
    ))

# 4. Global chart presentation layout
fig.update_layout(
    title=dict(
        text='Final Exam Score Breakdown: AI Tool Usage vs. AP Biology Background<br>'
             '<sup>Numbers above columns indicate group mean values</sup>',
        x=0.5
    ),
    xaxis=dict(
        title='AP Biology Background',
        categoryorder='array',
        categoryarray=display_names
    ),
    yaxis=dict(
        title='Final Exam Final Score',
        gridcolor='#eeeeee'
    ),
    boxmode='group', # Forces the 'Yes' and 'No' categories to sit side-by-side
    plot_bgcolor='white',
    showlegend=True,
    # FIXED LINE HERE: Changed 'orient' to 'orientation'
    legend=dict(title='AI Usage', orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=500
)

fig.show()

## Section 4: Does AI Help More with FRQ Than MCQ?

We compare how much AI usage is associated with performance
differences on MCQ and FRQ separately.

If AI is particularly useful for written reasoning tasks, we would
expect to see a larger difference between AI users and non-users
on FRQ than on MCQ.

In [32]:
# compute mean MCQ and FRQ scores by AI usage for each group
mcq_cols = ['MT1_MCQ', 'MT2_MCQ', 'MT3_MCQ']
frq_cols = ['MT1_FRQ', 'MT2_FRQ', 'MT3_FRQ']

df_mcqfrq = df_ai_clean.copy()

# normalize to percentages
for col in mcq_cols:
    df_mcqfrq[col] = df_mcqfrq[col] / 78 * 100
for col in frq_cols:
    df_mcqfrq[col] = df_mcqfrq[col] / 9 * 100

# average across all three midterms for each type
df_mcqfrq['Mean_MCQ'] = df_mcqfrq[mcq_cols].mean(axis=1)
df_mcqfrq['Mean_FRQ'] = df_mcqfrq[frq_cols].mean(axis=1)

# summary by AI usage — MCQ and FRQ separately, no gap calculation
summary = df_mcqfrq.groupby('Q133')[['Mean_MCQ', 'Mean_FRQ']].mean().round(2)
summary.index = ['No AI', 'Uses AI']
summary.loc['Difference (AI - No AI)'] = summary.loc['Uses AI'] - summary.loc['No AI']
print(summary)

                         Mean_MCQ  Mean_FRQ
No AI                       83.06     77.40
Uses AI                     83.34     76.84
Difference (AI - No AI)      0.28     -0.56


**Overall (all groups):**
- MCQ: AI users scored 0.26 points higher than non-users (83.34% vs 83.08%)
- FRQ: AI users scored 0.61 points *lower* than non-users (76.84% vs 77.45%)

Neither difference is meaningful — AI usage shows essentially no
association with MCQ or FRQ performance across the full cohort.

In [33]:
group_order   = ['1', '2', '3', '4', '5', 'No Background']
display_names = ['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']

results_mcqfrq = []
for group, display in zip(group_order, display_names):
    subset = df_mcqfrq[df_mcqfrq['Analysis_Group'] == group]
    for usage in ['Yes', 'No']:
        grp = subset[subset['Q133'] == usage]
        if len(grp) < 3:
            continue
        results_mcqfrq.append({
            'Group': display,
            'AI Usage': 'Uses AI' if usage == 'Yes' else 'No AI',
            'Mean MCQ': round(grp['Mean_MCQ'].mean(), 2),
            'Mean FRQ': round(grp['Mean_FRQ'].mean(), 2),
            'n': len(grp)
        })

df_mcqfrq_results = pd.DataFrame(results_mcqfrq)
df_mcqfrq_results

,Group,AI Usage,Mean MCQ,Mean FRQ,n
0,AP 1,No AI,68.67,59.81,5
1,AP 2,Uses AI,77.52,65.19,10
2,AP 2,No AI,72.64,56.96,15
3,AP 3,Uses AI,84.84,79.23,28
4,AP 3,No AI,80.45,75.10,29
5,AP 4,Uses AI,86.55,87.10,25
6,AP 4,No AI,86.59,86.24,44
7,AP 5,Uses AI,91.65,88.56,14
8,AP 5,No AI,92.14,92.21,28
9,No Background,Uses AI,81.22,72.48,85


In [34]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MCQ % Correct by AI Usage', 'FRQ % Correct by AI Usage'],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for col_idx, metric in enumerate(['Mean MCQ', 'Mean FRQ'], start=1):
    for usage, color in [('Uses AI', '#5DCAA5'), ('No AI', '#7FB3D3')]:
        subset = df_mcqfrq_results[df_mcqfrq_results['AI Usage'] == usage]
        fig.add_trace(
            go.Bar(
                name=usage,
                x=subset['Group'],
                y=subset[metric],
                marker_color=color,
                text=[f"{v:.1f}%" for v in subset[metric]],
                textposition='outside',
                showlegend=(col_idx == 1),
                legendgroup=usage,
                hovertemplate=f"<b>%{{x}} — {usage}</b><br>{metric}: %{{y:.1f}}%<extra></extra>"
            ),
            row=1, col=col_idx
        )

fig.update_layout(
    barmode='group',
    title=dict(
        text='MCQ vs FRQ Performance by AI Usage and AP Background<br>'
             '<sup>Does AI correlate more with FRQ than MCQ performance?</sup>',
        x=0.5
    ),
    yaxis=dict(title='Mean % Correct', range=[40, 105], gridcolor='#eeeeee'),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    plot_bgcolor='white',
    height=480
)

# fix ordering on both x-axes
fig.update_xaxes(
    categoryorder='array',
    categoryarray=['AP 1', 'AP 2', 'AP 3', 'AP 4', 'AP 5', 'No Background']
)

fig.show()

**AP 2 students show the most interesting pattern:**
- MCQ: AI users scored 4.88 points higher (77.52% vs 72.64%)
- FRQ: AI users scored 8.23 points higher (65.19% vs 56.96%)

The FRQ advantage for AP 2 AI users is nearly double the MCQ
advantage, suggesting AI may be more beneficial for written
reasoning than multiple choice recall among underprepared students.
Neither result is statistically significant given n=10/15, but the
directional pattern is consistent across the analysis.

**AP 4 and AP 5 students:** Virtually no difference between AI
users and non-users on either MCQ or FRQ, reinforcing that highly
prepared students do not benefit from AI tool usage regardless of
question format.

## Section 5: Does the MCQ-FRQ Gap Narrow Over the Semester?

We examine whether AI users show improvement in FRQ relative to MCQ
across the three midterms — a narrowing gap would suggest AI is
building written reasoning skills over time rather than providing
a static boost.

In [35]:
# compute MCQ-FRQ gap for each midterm separately, by AI usage
midterm_pairs = [
    ('MT1_MCQ', 'MT1_FRQ', 'Midterm 1'),
    ('MT2_MCQ', 'MT2_FRQ', 'Midterm 2'),
    ('MT3_MCQ', 'MT3_FRQ', 'Midterm 3'),
]

gap_results = []
for mcq_col, frq_col, label in midterm_pairs:
    for usage in ['Yes', 'No']:
        grp = df_mcqfrq[df_mcqfrq['Q133'] == usage]
        mcq_mean = (grp[mcq_col].mean())
        frq_mean = (grp[frq_col].mean())
        gap_results.append({
            'Midterm': label,
            'AI Usage': 'Uses AI' if usage == 'Yes' else 'No AI',
            'MCQ %': round(mcq_mean, 2),
            'FRQ %': round(frq_mean, 2),
        })

df_gaps = pd.DataFrame(gap_results)
df_gaps

,Midterm,AI Usage,MCQ %,FRQ %
0,Midterm 1,Uses AI,85.83,72.33
1,Midterm 1,No AI,85.52,75.48
2,Midterm 2,Uses AI,86.57,79.81
3,Midterm 2,No AI,85.76,79.36
4,Midterm 3,Uses AI,77.62,78.40
5,Midterm 3,No AI,77.89,77.37


In [36]:
# does the gap narrow over time specifically for AP 2 students?
gap_by_group = []
for mcq_col, frq_col, label in midterm_pairs:
    for group, display in zip(group_order, display_names):
        for usage in ['Yes', 'No']:
            grp = df_mcqfrq[
                (df_mcqfrq['Analysis_Group'] == group) &
                (df_mcqfrq['Q133'] == usage)
            ]
            if len(grp) < 3:
                continue
            mcq_mean = grp[mcq_col].mean()
            frq_mean = grp[frq_col].mean()
            gap_by_group.append({
                'Midterm': label,
                'Group': display,
                'AI Usage': 'Uses AI' if usage == 'Yes' else 'No AI',
                'MCQ %': round(mcq_mean, 2),
                'FRQ %': round(frq_mean, 2),
                'MCQ-FRQ Gap': round(mcq_mean - frq_mean, 2),
                'n': len(grp)
            })

df_gap_group = pd.DataFrame(gap_by_group)

# pivot to show gap across midterms side by side
df_gap_pivot = df_gap_group.pivot_table(
    index=['Group', 'AI Usage'],
    columns='Midterm',
    values='MCQ-FRQ Gap'
).round(2)
df_gap_pivot['Trend (MT1→MT3)'] = (df_gap_pivot['Midterm 3'] - df_gap_pivot['Midterm 1']).round(2)
df_gap_pivot

Midterm                 Midterm 1  Midterm 2  Midterm 3  Trend (MT1→MT3)
Group         AI Usage                                                  
AP 1          No AI         11.00      15.13       0.43           -10.57
AP 2          No AI         23.65      16.78       6.61           -17.04
              Uses AI       24.58      11.37       1.07           -23.51
AP 3          No AI          9.56       5.22       1.29            -8.27
              Uses AI       10.09       8.26      -1.53           -11.62
AP 4          No AI          2.06       0.92      -1.92            -3.98
              Uses AI        5.06      -1.37      -5.35           -10.41
AP 5          No AI         -0.79       2.20      -1.60            -0.81
              Uses AI        4.75       2.78       1.76            -2.99
No Background No AI         13.88       8.01       0.96           -12.92
              Uses AI       17.25       8.77       0.19           -17.06

In [37]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MCQ Performance Over Semester', 'FRQ Performance Over Semester'],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for usage, color, dash in [('Uses AI', '#5DCAA5', 'solid'), ('No AI', '#7FB3D3', 'dot')]:
    subset = df_gaps[df_gaps['AI Usage'] == usage]

    # MCQ — left panel
    fig.add_trace(go.Scatter(
        x=subset['Midterm'],
        y=subset['MCQ %'],
        mode='lines+markers',
        name=usage,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=10),
        legendgroup=usage,
        showlegend=True,
        hovertemplate=f"<b>{usage}</b><br>%{{x}}<br>MCQ: %{{y:.1f}}%<extra></extra>"
    ), row=1, col=1)

    # FRQ — right panel
    fig.add_trace(go.Scatter(
        x=subset['Midterm'],
        y=subset['FRQ %'],
        mode='lines+markers',
        name=usage,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=10),
        legendgroup=usage,
        showlegend=False,
        hovertemplate=f"<b>{usage}</b><br>%{{x}}<br>FRQ: %{{y:.1f}}%<extra></extra>"
    ), row=1, col=2)

# shared y range — use same scale for both panels
fig.update_yaxes(range=[50, 100], ticksuffix='%', gridcolor='#eeeeee')
fig.update_xaxes(showgrid=False)

fig.update_layout(
    title=dict(
        text='MCQ vs FRQ Performance Over the Semester by AI Usage<br>',
        x=0.5
    ),
    yaxis=dict(title='Mean % Correct'),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    plot_bgcolor='white',
    height=440,
    width=850
)
fig.show()

**Overall (all groups):** Both AI users and non-users show nearly
identical MCQ performance across all three midterms. On FRQ, AI
users start slightly lower on Midterm 1 but converge with non-users
by Midterm 3. The overall signal is weak. The Midterm 3 dip in MCQ
for both groups likely reflects course-level difficulty or timing
rather than an AI effect.

### Section 6: AP Score 2 Deep Dive — MCQ vs FRQ Over the Semester

In [38]:
# filter to AP 2 students only
df_gaps_ap2 = []

for mcq_col, frq_col, label in midterm_pairs:
    for usage in ['Yes', 'No']:
        grp = df_mcqfrq[
            (df_mcqfrq['Analysis_Group'] == '2') &
            (df_mcqfrq['Q133'] == usage)
        ]
        if len(grp) < 3:
            continue
        mcq_mean = grp[mcq_col].mean()
        frq_mean = grp[frq_col].mean()
        df_gaps_ap2.append({
            'Midterm': label,
            'AI Usage': 'Uses AI' if usage == 'Yes' else 'No AI',
            'MCQ %': round(mcq_mean, 2),
            'FRQ %': round(frq_mean, 2),
            'n': len(grp)
        })

df_gaps_ap2 = pd.DataFrame(df_gaps_ap2)
print(df_gaps_ap2)

# plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MCQ Performance — AP 2 Only', 'FRQ Performance — AP 2 Only'],
    shared_yaxes=True,
    horizontal_spacing=0.08
)

for usage, color, dash in [('Uses AI', '#5DCAA5', 'solid'), ('No AI', '#7FB3D3', 'dot')]:
    subset = df_gaps_ap2[df_gaps_ap2['AI Usage'] == usage]

    # MCQ
    fig.add_trace(go.Scatter(
        x=subset['Midterm'],
        y=subset['MCQ %'],
        mode='lines+markers',
        name=usage,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=10),
        legendgroup=usage,
        showlegend=True,
        hovertemplate=f"<b>{usage}</b><br>%{{x}}<br>MCQ: %{{y:.1f}}%<extra></extra>"
    ), row=1, col=1)

    # FRQ
    fig.add_trace(go.Scatter(
        x=subset['Midterm'],
        y=subset['FRQ %'],
        mode='lines+markers',
        name=usage,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=10),
        legendgroup=usage,
        showlegend=False,
        hovertemplate=f"<b>{usage}</b><br>%{{x}}<br>FRQ: %{{y:.1f}}%<extra></extra>"
    ), row=1, col=2)

fig.update_yaxes(range=[40, 100], ticksuffix='%', gridcolor='#eeeeee')
fig.update_xaxes(showgrid=False)

fig.update_layout(
    title=dict(
        text='MCQ vs FRQ Performance Over Semester — AP Score 2 Students Only<br>'
             '<sup>n=10 AI users vs n=15 non-users</sup>',
        x=0.5
    ),
    yaxis=dict(title='Mean % Correct'),
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    plot_bgcolor='white',
    height=440,
    width=850
)
fig.show()

     Midterm AI Usage  MCQ %  FRQ %   n
0  Midterm 1  Uses AI  76.80  52.22  10
1  Midterm 1    No AI  76.80  53.15  15
2  Midterm 2  Uses AI  81.92  70.56  10
3  Midterm 2    No AI  73.08  56.30  15
4  Midterm 3  Uses AI  73.85  72.78  10
5  Midterm 3    No AI  68.05  61.44  15


**MCQ:** Non-AI users start slightly higher (77% vs 76%) but decline
steadily to 68% by Midterm 3, while AI users maintain performance
better, ending at 74%. The gap widens in favor of AI users as the
semester progresses.

**FRQ:** Both groups start nearly identical (~53% on Midterm 1). AI
users then improve dramatically to 71% by Midterm 2 (+18 points)
while non-AI users gain only modestly (+3 points). The gap persists
through Midterm 3 (73% vs 62%).

The FRQ trajectory is the strongest signal in the AI analysis —
suggesting AI tools may help underprepared students build written
reasoning skills over the semester rather than providing a one-time
boost.

**Caveat:** n=10 AI users vs n=15 non-users. Individual students
have an outsized effect on these results. Treat as exploratory and
prioritize replication with a larger sample.